# Capítulo 8 — Métodos de classificação

**Livro:** começa na página **159** do PDF.

📗 **Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
set.seed(42)
options(digits = 4, repr.plot.width = 7, repr.plot.height = 4.5)
suppressPackageStartupMessages({ library(glmnet); library(rpart); library(randomForest); library(FNN); library(e1071) })
cat("Pronto.", R.version.string, "
")

### Regressão logística · PDF p. 160

#### 🧪 Exercício: Regressão logística

Com `mtcars`, ajuste `glm(am ~ wt + hp, family = binomial)`. Preveja a probabilidade de câmbio manual de um carro com wt = 2,5 e hp = 120 e calcule a acurácia na base com corte 0,5.

In [ ]:
m <- glm(am ~ wt + hp, data = mtcars, family = binomial)
predict(m, data.frame(wt = 2.5, hp = 120), type = "response")
mean((fitted(m) > 0.5) == mtcars$am)

### Bayes Ingênuo · PDF p. 162

#### 🧪 Exercício: Naive Bayes

Separe `iris` em treino (100) e teste (50) e ajuste `naiveBayes(Species ~ .)`. Mostre a matriz de confusão no teste.

In [ ]:
i <- sample(150, 100)
m <- naiveBayes(Species ~ ., iris[i, ])
table(previsto = predict(m, iris[-i, ]), real = iris$Species[-i])

### Análise Discriminante · PDF p. 165

#### 🧪 Exercício: LDA × QDA

No mesmo treino/teste da `iris`, compare a acurácia de teste de `MASS::lda` e `MASS::qda`.

In [ ]:
library(MASS)
i <- sample(150, 100)
acc <- function(m) mean(predict(m, iris[-i, ])$class == iris$Species[-i])
c(LDA = acc(lda(Species ~ ., iris[i, ])), QDA = acc(qda(Species ~ ., iris[i, ])))

## Support Vector Machines (SVM) · PDF p. 169

#### 🧪 Exercício: SVM em dados circulares

Gere 400 pontos em [−1, 1]² com classe "dentro" se x² + y² < 0,5. Compare SVM com núcleo linear e radial (acurácia em 2.000 pontos de teste). Por que o linear falha?

In [ ]:
gera <- function(n) { x <- runif(n, -1, 1); y <- runif(n, -1, 1); data.frame(x, y, classe = factor(x^2 + y^2 < 0.5)) }
tr <- gera(400); te <- gera(2000)
acc <- function(k) mean(predict(svm(classe ~ ., tr, kernel = k), te) == te$classe)
c(linear = acc("linear"), radial = acc("radial"))  # nenhuma reta separa um círculo

## Árvores de Classificação · PDF p. 174

#### 🧪 Exercício: Árvore de classificação

Ajuste `rpart(Species ~ ., iris)`, desenhe a árvore e conte quantas regras (folhas) ela tem.

In [ ]:
arv <- rpart(Species ~ ., iris)
plot(arv, margin = 0.1); text(arv, use.n = TRUE, cex = 0.8)
sum(arv$frame$var == "<leaf>")

## Método dos k Vizinhos Mais Próximos · PDF p. 179

#### 🧪 Exercício: KNN com padronização

Use `FNN::knn` na `iris` (treino 100, teste 50) com k = 1, 5, 15 e 45, **padronizando** as variáveis com `scale` (média e desvio do treino). Qual k acerta mais?

In [ ]:
i <- sample(150, 100)
tr <- scale(iris[i, 1:4]); te <- scale(iris[-i, 1:4], attr(tr, "scaled:center"), attr(tr, "scaled:scale"))
sapply(c(1, 5, 15, 45), function(k) mean(FNN::knn(tr, te, iris$Species[i], k = k) == iris$Species[-i]))

## Exemplos · PDF p. 181

#### 🧪 Exercício: Comparando classificadores

No problema dos círculos, monte uma tabela com a acurácia de teste de: logística, naive Bayes, árvore, floresta, KNN (k = 15) e SVM radial.

In [ ]:
gera <- function(n) { x <- runif(n, -1, 1); y <- runif(n, -1, 1); data.frame(x, y, classe = factor(x^2 + y^2 < 0.5)) }
tr <- gera(500); te <- gera(2000)
prev <- list(
  logistica = ifelse(predict(glm(classe ~ ., binomial, data = tr), te, type = "response") > 0.5, "TRUE", "FALSE"),
  naive_bayes = predict(naiveBayes(classe ~ ., tr), te),
  arvore = predict(rpart(classe ~ ., tr), te, type = "class"),
  floresta = predict(randomForest(classe ~ ., tr), te),
  knn15 = FNN::knn(tr[1:2], te[1:2], tr$classe, k = 15),
  svm_radial = predict(svm(classe ~ ., tr), te))
sort(sapply(prev, function(p) mean(as.character(p) == as.character(te$classe))), decreasing = TRUE)